# 04 · Embeddings MiniLM: congelados y con ajuste fino

Pregunta P2: ¿cuánto mejora una representación contextual respecto de TF-IDF, y cuánto aporta ajustarla a la tarea?

Se usa **un solo encoder** en las dos variantes, `all-MiniLM-L6-v2` (6 capas, 22 millones de parámetros), para que la diferencia entre ellas sea solo el ajuste y no el modelo:

| Variante | Qué se entrena | Dónde se evalúa |
|---|---|---|
| **Congelado** | Nada del encoder. Se guarda el vector de cada letra (384 números); la regresión logística se entrena después, en `src/resultados.py` | 5 particiones |
| **Ajuste fino** | Todo el encoder más una capa lineal de 5 salidas | 5 particiones (`PARTICIONES`) |

En las dos, cada letra se recorta a sus primeros 256 tokens (el largo con el que se entrenó el modelo, unas 190 palabras) y su vector es el promedio de los vectores de sus tokens.

Las particiones son las de `folds.parquet`, agrupadas por artista. La época del ajuste fino se elige con una validación interna (15% de los artistas del entrenamiento); la partición de evaluación no se usa para decidir nada. La tasa de aprendizaje y el tamaño de lote están fijos de antemano.

**Cómo correrlo en Colab.** Entorno de ejecución → Cambiar tipo de entorno → GPU T4. Ejecutar todo. La primera celda pide subir `letras_colab.parquet` (lo genera `python -m src.resultados --exportar`). Al final se descarga `salida_minilm.zip`, que se descomprime en `data/processed/`. Tarda alrededor de 30 minutos con las 5 particiones; cada partición se guarda al terminar, así que se puede interrumpir y retomar.

In [ ]:
import json
import os
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupShuffleSplit
from torch import nn
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

MODELO = "sentence-transformers/all-MiniLM-L6-v2"
GENEROS = ["pop", "rap", "rock", "rb", "country"]
SEMILLA = 42
MAX_TOKENS = 256
PARTICIONES = [0, 1, 2, 3, 4]          # para correr una sola: [0]
EPOCAS, LOTE, TASA, DECAIMIENTO = 4, 32, 5e-5, 0.01
PRUEBA = os.environ.get("LYRICS_PRUEBA") == "1"   # corrida mínima para verificar que el código funciona

SALIDA = Path("salida_prueba" if PRUEBA else "salida")
SALIDA.mkdir(exist_ok=True)
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", dispositivo, torch.cuda.get_device_name(0) if dispositivo == "cuda" else "")

candidatos = [Path("letras_colab.parquet"), Path("data/processed/letras_colab.parquet"),
              Path("../data/processed/letras_colab.parquet")]
DATOS = next((p for p in candidatos if p.exists()), None)
if DATOS is None:
    from google.colab import files      # en Colab: elegir letras_colab.parquet
    files.upload()
    DATOS = candidatos[0]

d = pd.read_parquet(DATOS)
if PRUEBA:
    d = d.groupby("tag").head(120).reset_index(drop=True)
    MAX_TOKENS, EPOCAS, PARTICIONES = 64, 2, [0, 1]
y = pd.Categorical(d["tag"], categories=GENEROS).codes.astype(int)
fold = d["fold"].to_numpy()
artistas = d["artist"].to_numpy()
print(f"{len(d):,} canciones de {d['artist'].nunique():,} artistas")

## 1. Tokens

Cada letra se convierte una sola vez en su lista de tokens, recortada a `MAX_TOKENS`.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODELO)
t0 = time.time()
ids = tok(d["letra"].tolist(), truncation=True, max_length=MAX_TOKENS)["input_ids"]
largo = np.array([len(x) for x in ids])
print(f"{time.time() - t0:.0f} s · letras recortadas: {(largo == MAX_TOKENS).mean():.0%}")
print(pd.Series(largo == MAX_TOKENS).groupby(d["tag"]).mean().round(2).to_dict())


def lote(indices):
    """Tensores de tokens y máscara de un grupo de canciones, rellenados al largo de la más larga."""
    n = max(len(ids[i]) for i in indices)
    x = torch.full((len(indices), n), tok.pad_token_id, dtype=torch.long)
    m = torch.zeros((len(indices), n), dtype=torch.long)
    for j, i in enumerate(indices):
        x[j, :len(ids[i])] = torch.tensor(ids[i])
        m[j, :len(ids[i])] = 1
    return x.to(dispositivo), m.to(dispositivo)

## 2. Modelo

El encoder devuelve un vector por token. El vector de la letra es el promedio de los vectores de sus tokens (sin contar el relleno), igual que en `sentence-transformers`. Para clasificar se agrega una capa lineal de 5 salidas, una por género.

In [ ]:
class Clasificador(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODELO)
        self.salida = nn.Linear(self.encoder.config.hidden_size, len(GENEROS))

    def representar(self, x, m):
        h = self.encoder(input_ids=x, attention_mask=m).last_hidden_state
        m = m.unsqueeze(-1).to(h.dtype)
        return (h * m).sum(1) / m.sum(1)

    def forward(self, x, m):
        return self.salida(self.representar(x, m))


@torch.no_grad()
def aplicar(funcion, indices, tam=128):
    """Aplica `funcion(x, m)` a las canciones `indices` y devuelve el resultado en ese mismo orden."""
    indices = np.asarray(indices)
    orden = np.argsort(largo[indices], kind="stable")      # lotes de largo parecido: menos relleno
    partes = []
    for a in range(0, len(indices), tam):
        x, m = lote(indices[orden[a:a + tam]])
        with torch.autocast(device_type=dispositivo, enabled=dispositivo == "cuda"):
            partes.append(funcion(x, m).float().cpu())
    salida = torch.cat(partes).numpy()
    resultado = np.empty_like(salida)
    resultado[orden] = salida
    return resultado

## 3. Embeddings congelados

El modelo tal como se descarga, sin entrenar nada. Los vectores se normalizan a norma 1 y se guardan para entrenar la regresión logística y calcular la pureza de vecinos fuera de este notebook.

In [ ]:
ruta_emb = SALIDA / "emb_minilm_congelado.npz"
if not ruta_emb.exists():
    t0 = time.time()
    base = Clasificador().to(dispositivo).eval()
    emb = aplicar(base.representar, np.arange(len(d)))
    emb /= np.linalg.norm(emb, axis=1, keepdims=True)
    np.savez_compressed(ruta_emb, id=d["id"].to_numpy(), emb=emb.astype(np.float16))
    del base
    print(f"embeddings {emb.shape} en {time.time() - t0:.0f} s")

## 4. Ajuste fino

Para cada partición: se entrena con las otras cuatro (menos el 15% de artistas de la validación interna), se mide el macro-F1 en la validación interna al final de cada época, se conserva la mejor época y con ese modelo se predice la partición de evaluación.

In [ ]:
def afinar(k):
    torch.manual_seed(SEMILLA)
    rng = np.random.default_rng(SEMILLA)
    i_te, i_tr = np.flatnonzero(fold == k), np.flatnonzero(fold != k)
    a, b = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEMILLA).split(i_tr, groups=artistas[i_tr]))
    i_fit, i_val = i_tr[a], i_tr[b]

    modelo = Clasificador().to(dispositivo)
    opt = torch.optim.AdamW(modelo.parameters(), lr=TASA, weight_decay=DECAIMIENTO)
    pasos = EPOCAS * int(np.ceil(len(i_fit) / LOTE))
    plan = get_linear_schedule_with_warmup(opt, int(0.1 * pasos), pasos)
    escala = torch.amp.GradScaler(enabled=dispositivo == "cuda")
    perdida = nn.CrossEntropyLoss()
    yt = torch.tensor(y)

    mejor, estado, f1_val, t0 = -1.0, None, [], time.time()
    for epoca in range(EPOCAS):
        modelo.train()
        mezcla = rng.permutation(i_fit)
        for p in range(0, len(mezcla), LOTE):
            idx = mezcla[p:p + LOTE]
            x, m = lote(idx)
            with torch.autocast(device_type=dispositivo, enabled=dispositivo == "cuda"):
                costo = perdida(modelo(x, m).float(), yt[idx].to(dispositivo))
            opt.zero_grad()
            escala.scale(costo).backward()
            escala.step(opt)
            escala.update()
            plan.step()
        modelo.eval()
        f1 = f1_score(y[i_val], aplicar(modelo, i_val).argmax(1), average="macro")
        f1_val.append(round(float(f1), 4))
        print(f"  partición {k} época {epoca + 1}: macro-F1 de validación interna {f1:.3f} ({time.time() - t0:.0f} s)")
        if f1 > mejor:
            mejor, estado = f1, {n: v.detach().cpu().clone() for n, v in modelo.state_dict().items()}

    modelo.load_state_dict(estado)
    modelo.eval()
    proba = torch.softmax(torch.tensor(aplicar(modelo, i_te)), 1).numpy()
    f1_te = f1_score(y[i_te], proba.argmax(1), average="macro")
    print(f"partición {k}: macro-F1 {f1_te:.3f} (época elegida: {int(np.argmax(f1_val)) + 1})")
    tabla = pd.DataFrame(proba, columns=[f"p_{g}" for g in GENEROS])
    tabla.insert(0, "fold", k)
    tabla.insert(0, "id", d["id"].to_numpy()[i_te])
    tabla["pred"] = np.array(GENEROS)[proba.argmax(1)]
    return tabla, {"particion": k, "macro_f1": round(float(f1_te), 4), "f1_validacion_por_epoca": f1_val,
                   "epoca_elegida": int(np.argmax(f1_val)) + 1, "segundos": round(time.time() - t0)}


for k in PARTICIONES:
    ruta = SALIDA / f"oof_particion{k}.parquet"
    if ruta.exists():
        print(f"partición {k}: ya estaba hecha")
        continue
    tabla, resumen = afinar(k)
    tabla.to_parquet(ruta, index=False)
    (SALIDA / f"resumen_particion{k}.json").write_text(json.dumps(resumen))

## 5. Salida

Un archivo con las predicciones fuera de muestra de todas las particiones corridas, con el mismo formato que el resto de los modelos (`data/processed/oof/`), y un resumen con los hiperparámetros y el macro-F1 de cada partición.

In [ ]:
hechas = sorted(SALIDA.glob("oof_particion*.parquet"))
oof = pd.concat([pd.read_parquet(p) for p in hechas], ignore_index=True)
oof.to_parquet(SALIDA / "M5_minilm_afinado.parquet", index=False)
resumen = {"modelo": MODELO, "max_tokens": MAX_TOKENS, "epocas": EPOCAS, "lote": LOTE, "tasa": TASA,
           "decaimiento": DECAIMIENTO, "calentamiento": 0.1, "dispositivo": dispositivo,
           "particiones": [json.loads(p.read_text()) for p in sorted(SALIDA.glob("resumen_particion*.json"))]}
(SALIDA / "M5_minilm_afinado.json").write_text(json.dumps(resumen, indent=1))
f1s = [r["macro_f1"] for r in resumen["particiones"]]
print(f"ajuste fino: macro-F1 {np.mean(f1s):.3f} ± {np.std(f1s):.3f} en {len(f1s)} particiones, {len(oof):,} canciones")

with zipfile.ZipFile("salida_minilm.zip", "w", zipfile.ZIP_STORED) as z:
    for nombre in ["emb_minilm_congelado.npz", "M5_minilm_afinado.parquet", "M5_minilm_afinado.json"]:
        z.write(SALIDA / nombre, nombre)
try:
    from google.colab import files
    files.download("salida_minilm.zip")
except ImportError:
    print("listo: salida_minilm.zip")